In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import scanpy as sc

In [ ]:
path = "data/xenium/processed/04-kidney_tcr_tsub.h5ad"
adata = sc.read_h5ad(path)
adata.X = adata.layers["counts"].copy()

tcell_keys = ["CD4+", "TFH", "Tregs", "MAIT", "CD8+", "NK/NKT", "gdT"]

assert all(tcell_key in adata.obs["tcell_subtype"].unique() for tcell_key in tcell_keys)

# define broad and fine cell types
adata.obs["cell_type_l1"] = adata.obs["cell_type_no_tcr"].astype(str)
t_mask = adata.obs["cell_type_l1"] == "T"
adata.obs.loc[t_mask, "cell_type_l1"] = adata.obs.loc[t_mask, "tcell_subtype"]
adata.obs.loc[adata.obs["cell_type_l1"].isin(tcell_keys), "cell_type_l1"] = "T"


adata.obs["cell_type_l2"] = adata.obs["tcell_subtype"].copy()

# merge TFH with CD4+
mapping = {
    "TFH": "CD4+",
    "NK/NKT": "NKT-like",
}
adata.obs["cell_type_l2"] = (
    adata.obs["cell_type_l2"].astype(str).replace(mapping).astype("category")
)


adata

In [ ]:
label_key = "cell_type_l2"
# label_key = "cell_type_l1"


adata.obs[label_key].value_counts()

In [ ]:
label_key = "cell_type_l2"
# label_key = "cell_type_l1"


adata.obs[label_key].value_counts()

In [ ]:
adata.write_h5ad("data/xenium/processed/04.1-kidney_tcr_tsub_harmonized.h5ad")